# FreshCart | From raw events to Gold
Parts B1–B5 and bonus. Classroom sample data and Snowflake SQL retained. The local cross-check below actually runs in Python; it is **not** evidence of Snowflake execution. All cloud cells are opt-in, and no credentials or copied screenshots are embedded.

For live work: install `snowflake-connector-python`, `python-dotenv` and `pandas` in your notebook kernel. Keep `.env` outside Git. In Colab use a supported noninteractive authentication method, such as securely configured key-pair or programmatic token; browser authentication should be tried on your local machine. Review and complete your own B4 predictions before running that experiment.

In [1]:
# Change only when the corresponding prerequisites are satisfied.
RUN_LIVE = False
B4_PREDICTIONS_RECORDED = False
RUN_BONUS = False
BONUS_BATCH_FILE = 'bonus_batch.jsonl'
conn = None

## Classroom fixtures

In [2]:
# Define deliberately messy sample API data so we can demonstrate cleaning, deduplication, and enrichment.
# ---- Website orders: flat JSON ----
MOCK_WEB_ORDERS = [
    # ORD-1001: email has spaces and capitals, amount has a comma
    {"order_id": "ORD-1001", "cust_email": " Priya.R@Gmail.com ", "cust_name": "priya r",
     "order_date": "2026-09-01", "amount": "1,250.00", "currency": "INR", "pincode": "600041",
     "updated_at": "2026-09-01 10:02:00"},
    # ORD-1003: amount is missing (the mobile copy below has it)
    {"order_id": "ORD-1003", "cust_email": "meera@outlook.com", "cust_name": "MEERA S",
     "order_date": "2026-09-02", "amount": None, "currency": "INR", "pincode": "400050",
     "updated_at": "2026-09-02 09:10:00"},
    # ORD-1002: a dollar order (also sent by the mobile app = duplicate)
    {"order_id": "ORD-1002", "cust_email": "Arjun.K@Yahoo.in ", "cust_name": "Arjun K",
     "order_date": "2026-09-01", "amount": "42.50", "currency": "USD", "pincode": "560034",
     "updated_at": "2026-09-01 11:15:00"},
    # ORD-1004: negative amount -> will be rejected
    {"order_id": "ORD-1004", "cust_email": "rahul.m@gmail.com", "cust_name": "Rahul M",
     "order_date": "Sep 3, 2026", "amount": "-500", "currency": "INR", "pincode": "110001",
     "updated_at": "2026-09-03 12:00:00"},
    # ORD-1005: rupee sign in the amount, lowercase currency, dot after the name
    {"order_id": "ORD-1005", "cust_email": "priya.r@gmail.com", "cust_name": "Priya R.",
     "order_date": "2026-09-03", "amount": "₹ 899", "currency": "inr", "pincode": "600041",
     "updated_at": "2026-09-03 16:45:00"},
    # ORD-1006: bad email and 5-digit pincode -> will be rejected
    {"order_id": "ORD-1006", "cust_email": "vikram@@mail", "cust_name": "Vikram",
     "order_date": "2026-09-04", "amount": 30, "currency": "EUR", "pincode": "99999",
     "updated_at": "2026-09-04 08:30:00"},
]

# ---- Mobile app orders: nested JSON, different field names, day-first dates ----
MOCK_MOBILE_ORDERS = [
    # ORD-1002 again (same as the web copy)
    {"orderId": "ORD-1002", "customer": {"email": "arjun.k@yahoo.in", "name": "Arjun K"},
     "orderDate": "01/09/2026", "total": {"value": "42.50", "currency": "usd"},
     "deliveryPin": "560034", "updatedAt": "2026-09-01T11:15:00"},
    # ORD-1003 again, newer and with the amount filled in
    {"orderId": "ORD-1003", "customer": {"email": "Meera@Outlook.com", "name": "Meera S"},
     "orderDate": "02/09/2026", "total": {"value": "2,100.00", "currency": "INR"},
     "deliveryPin": "400050", "updatedAt": "2026-09-02T18:40:00"},
]

# ---- Customer tier changes (Priya moves from Silver to Gold on 3 Sept) ----
MOCK_CRM_TIER_EVENTS = [
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Silver", "effective_from": "2023-04-10"},
    {"cust_id": "CUST-01", "email": "priya.r@gmail.com", "tier": "Gold", "effective_from": "2026-09-03"},
    {"cust_id": "CUST-02", "email": "arjun.k@yahoo.in", "tier": "Silver", "effective_from": "2024-01-15"},
    {"cust_id": "CUST-03", "email": "meera@outlook.com", "tier": "Platinum", "effective_from": "2022-11-01"},
]

# ---- Exchange rates: rupees per 1 US dollar, per day ----
MOCK_FX_RESPONSES = [
    {"amount": 1.0, "base": "USD", "start_date": "2026-09-01", "end_date": "2026-09-03",
     "rates": {"2026-09-01": {"INR": 83.20}, "2026-09-02": {"INR": 83.35}, "2026-09-03": {"INR": 83.10}}},
]

# ---- Pincodes: pincode -> (city, state) ----
MOCK_PINCODES = {"600041": ("Chennai", "Tamil Nadu"), "560034": ("Bengaluru", "Karnataka"),
                 "400050": ("Mumbai", "Maharashtra")}


def mock_pincode_response(pin):
    """Reply in the same shape the India Post API uses."""
    if pin not in MOCK_PINCODES:
        return [{"Message": "No records found", "Status": "Error", "PostOffice": None}]
    district, state = MOCK_PINCODES[pin]
    return [{"Message": "Number of pincode(s) found:1", "Status": "Success",
             "PostOffice": [{"Name": district, "District": district, "State": state,
                             "Country": "India", "Pincode": pin}]}]

## B1 — local calculation and expected baseline

In [3]:
from decimal import Decimal
import re
from datetime import datetime

def compute_baseline(currency_rule=False):
    rows = []
    for kind, records in [('web', MOCK_WEB_ORDERS), ('mobile', MOCK_MOBILE_ORDERS)]:
        for r in records:
            if kind == 'web':
                oid, email, ds, raw, cur, pin, stamp = (r[k] for k in ('order_id','cust_email','order_date','amount','currency','pincode','updated_at'))
            else:
                oid,email,ds,raw,cur,pin,stamp = r['orderId'],r['customer']['email'],r['orderDate'],r['total']['value'],r['total']['currency'],r['deliveryPin'],r['updatedAt']
            email=email.strip().lower();cur=cur.upper();amount=None if raw is None else Decimal(re.sub(r'[^0-9.-]','',str(raw)))
            for fmt in ('%Y-%m-%d','%d/%m/%Y','%b %d, %Y'):
                try: day=datetime.strptime(ds,fmt).date().isoformat();break
                except ValueError: pass
            reasons=[]
            if not re.fullmatch(r'[^@\s]+@[^@\s]+\.[a-z]{2,}',email):reasons.append('bad_email')
            if not re.fullmatch(r'[1-9][0-9]{5}',pin):reasons.append('bad_pincode')
            if amount is not None and amount<=0:reasons.append('non_positive_amount')
            if currency_rule and cur not in {'INR', 'USD'}: reasons.append('unsupported_currency')
            rows.append(dict(order_id=oid,email=email,day=day,amount=amount,currency=cur,updated_at=datetime.fromisoformat(stamp),source=kind,reasons=reasons))
    clean=[r for r in rows if not r['reasons']];latest={}
    for r in clean:
        if r['order_id'] not in latest or r['updated_at']>=latest[r['order_id']]['updated_at']:latest[r['order_id']]=r
    gold=[]
    for oid,r in sorted(latest.items()):
        tier=max((v for v in MOCK_CRM_TIER_EVENTS if v['email']==r['email'] and v['effective_from']<=r['day']),key=lambda v:v['effective_from'])['tier']
        rate=Decimal('1') if r['currency']=='INR' else Decimal(str(MOCK_FX_RESPONSES[0]['rates'][r['day']]['INR']))
        amount=(r['amount']*rate).quantize(Decimal('.01'));pts=int(amount//100)*{'Silver':1,'Gold':2,'Platinum':3}[tier]
        gold.append(dict(order_id=oid,tier=tier,amount_inr=str(amount),loyalty_points=pts))
    summary={'bronze':len(rows),'clean':len(clean),'quarantine':len(rows)-len(clean),'dedup':len(gold),'revenue_inr':str(sum(Decimal(x['amount_inr']) for x in gold)),'points':sum(x['loyalty_points'] for x in gold),'rejected_percent':100*(len(rows)-len(clean))/len(rows)}
    return summary,gold,rows

In [4]:
import pandas as pd
baseline_local, local_gold, local_rows = compute_baseline()
strict_local, strict_gold, strict_rows = compute_baseline(currency_rule=True)
assert (baseline_local['bronze'],baseline_local['clean'],baseline_local['quarantine'],baseline_local['dedup']) == (8,6,2,4)
assert baseline_local['revenue_inr']=='7785.00' and baseline_local['points']==126
assert baseline_local==strict_local
print('LOCAL PYTHON CROSS-CHECK (not Snowflake)')
display(pd.DataFrame([baseline_local]))
display(pd.DataFrame(local_gold))
print('B3 local validation comparison:')
for label, records in [('before',local_rows),('after',strict_rows)]:
    print(label, next(r['reasons'] for r in records if r['order_id']=='ORD-1006'))

LOCAL PYTHON CROSS-CHECK (not Snowflake)


,bronze,clean,quarantine,dedup,revenue_inr,points,rejected_percent
0,8,6,2,4,7785.00,126,25.0


,order_id,tier,amount_inr,loyalty_points
0,ORD-1001,Silver,1250.00,12
1,ORD-1002,Silver,3536.00,35
2,ORD-1003,Platinum,2100.00,63
3,ORD-1005,Gold,899.00,16


B3 local validation comparison:
before ['bad_email', 'bad_pincode']
after ['bad_email', 'bad_pincode', 'unsupported_currency']


## Raw landing and Snowflake helpers

In [5]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from tempfile import mkdtemp
import gzip, json, re
import pandas as pd
LANDING_DIR = Path(mkdtemp(prefix='freshcart_landing_'))


def write_landing(source, records, run_id):
    folder = LANDING_DIR / source
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f'{source}__{run_id}.json.gz'
    stamp = datetime.now(timezone.utc).isoformat()
    with gzip.open(path, 'wt', encoding='utf8') as file:
        for item in records:
            file.write(json.dumps(dict(source=source, run_id=run_id, extracted_at=stamp, data=item), ensure_ascii=False) + '\n')
    return path


def run_sql(sql):
    conn.execute_string(sql, remove_comments=True)


def query(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params) if params is not None else cur.execute(sql)
        return cur.fetchall()


def show(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])


def load_file(local_path, table, stage_dir):
    assert table in {'bronze.orders_raw','bronze.customers_raw','bronze.fx_raw','bronze.pincode_raw'}
    assert re.fullmatch(r'[A-Za-z0-9_/-]+', stage_dir)
    path = Path(local_path).resolve()
    uri = 'file://' + path.as_posix().replace("'", "''")
    query(f"PUT '{uri}' @bronze.landing/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=FALSE")
    rows = query(f"""COPY INTO {table} (source,raw,run_id,extracted_at,src_file,src_row)
       FROM (SELECT $1:source::STRING,$1:data,$1:run_id::STRING,
                    $1:extracted_at::TIMESTAMP_TZ,METADATA$FILENAME,METADATA$FILE_ROW_NUMBER
             FROM @bronze.landing/{stage_dir}/)
       FILE_FORMAT=(FORMAT_NAME='bronze.ff_ndjson') ON_ERROR='ABORT_STATEMENT'""")
    print(table, rows)
    return rows

### SQL 00 SETUP

In [6]:
SQL_00_SETUP = r"""
-- =====================================================================
-- Setup: creates the warehouse, database, schemas and empty tables.
-- Safe to re-run: everything is IF NOT EXISTS, except the two small seed tables.
-- =====================================================================
CREATE WAREHOUSE IF NOT EXISTS FRESHCART_WH
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE FRESHCART_WH;

CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS bronze;      -- raw API responses, unchanged
CREATE SCHEMA IF NOT EXISTS silver;      -- cleaned, typed, deduplicated
CREATE SCHEMA IF NOT EXISTS dim;         -- dimensions (customers)
CREATE SCHEMA IF NOT EXISTS ref;         -- reference data (fx, pincodes, tiers)
CREATE SCHEMA IF NOT EXISTS gold;        -- business-ready tables
CREATE SCHEMA IF NOT EXISTS quarantine;  -- rows that failed validation

-- Landing area inside Snowflake for the files Python uploads
CREATE FILE FORMAT IF NOT EXISTS bronze.ff_ndjson TYPE = JSON COMPRESSION = AUTO;
CREATE STAGE IF NOT EXISTS bronze.landing FILE_FORMAT = bronze.ff_ndjson;

-- ---------- BRONZE: one table per source, raw JSON in a VARIANT column ----------
CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  ingest_seq   NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER,   -- arrival order
  source       STRING,          -- 'web' or 'mobile'
  raw          VARIANT,         -- the API record exactly as received
  run_id       STRING,
  extracted_at TIMESTAMP_TZ,
  src_file     STRING,
  src_row      NUMBER,
  loaded_at    TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.fx_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);
CREATE TABLE IF NOT EXISTS bronze.pincode_raw (
  ingest_seq NUMBER AUTOINCREMENT START 1 INCREMENT 1 ORDER, source STRING, raw VARIANT,
  run_id STRING, extracted_at TIMESTAMP_TZ, src_file STRING, src_row NUMBER,
  loaded_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- ---------- REF: small hand-maintained seed tables ----------
CREATE OR REPLACE TABLE ref.tier_multiplier AS
SELECT * FROM VALUES ('Silver', 1), ('Gold', 2), ('Platinum', 3) AS v(tier, multiplier);

CREATE OR REPLACE TABLE ref.state_region AS
SELECT * FROM VALUES
  ('Tamil Nadu','South'), ('Karnataka','South'), ('Kerala','South'), ('Andhra Pradesh','South'),
  ('Telangana','South'), ('Puducherry','South'),
  ('Maharashtra','West'), ('Gujarat','West'), ('Goa','West'), ('Rajasthan','West'),
  ('Delhi','North'), ('Haryana','North'), ('Punjab','North'), ('Uttar Pradesh','North'),
  ('Himachal Pradesh','North'), ('Jammu and Kashmir','North'), ('Uttarakhand','North'),
  ('West Bengal','East'), ('Odisha','East'), ('Bihar','East'), ('Jharkhand','East'), ('Assam','East'),
  ('Madhya Pradesh','Central'), ('Chhattisgarh','Central')
  AS v(state, region);

-- Filled incrementally from the India Post API (see SQL_32_PINCODE_GEO)
CREATE TABLE IF NOT EXISTS ref.pincode_geo (
  pincode STRING PRIMARY KEY, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_LTZ
);

-- ---------- QUARANTINE, GOLD ----------
CREATE TABLE IF NOT EXISTS quarantine.orders (
  run_id STRING, quarantined_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
  ingest_seq NUMBER, source STRING, order_id STRING, reasons ARRAY, row_data VARIANT
);

CREATE TABLE IF NOT EXISTS gold.fct_orders (
  order_id STRING PRIMARY KEY, order_date DATE, cust_id STRING, email STRING,
  tier STRING, multiplier NUMBER, amount NUMBER(12,2), currency STRING,
  fx_rate NUMBER(12,4), amount_inr NUMBER(14,2), loyalty_points NUMBER,
  is_high_value BOOLEAN, city STRING, state STRING, region STRING,
  updated_at TIMESTAMP_NTZ, row_hash NUMBER, published_at TIMESTAMP_LTZ
);
"""

### SQL 10 ORDERS CLEAN

In [7]:
SQL_10_ORDERS_CLEAN = r"""
-- =====================================================================
-- SILVER 1: unify the two order APIs, cleanse, validate, quarantine.
-- Input : bronze.orders_raw (raw JSON from 'web' and 'mobile')
-- Output: silver.orders_checked (all rows + reasons), silver.orders_clean (good rows),
--         quarantine.orders (bad rows, appended once per row)
-- =====================================================================
CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
WITH unified AS (
  -- The two APIs name and nest fields differently; map both to one set of columns.
  SELECT ingest_seq, source,
         raw:order_id::STRING   AS order_id,
         raw:cust_email::STRING AS cust_email,
         raw:cust_name::STRING  AS cust_name,
         raw:order_date::STRING AS order_date,
         raw:amount::STRING     AS amount,
         raw:currency::STRING   AS currency,
         raw:pincode::STRING    AS pincode,
         raw:updated_at::STRING AS updated_at
  FROM bronze.orders_raw WHERE source = 'web'
  UNION ALL
  SELECT ingest_seq, source,
         raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
         raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
         raw:deliveryPin::STRING, raw:updatedAt::STRING
  FROM bronze.orders_raw WHERE source = 'mobile'
),
typed AS (
  SELECT
    ingest_seq, source,
    TRIM(order_id)                                    AS order_id,
    LOWER(TRIM(cust_email))                           AS email,
    INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
    COALESCE(
      TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
      TRY_TO_DATE(order_date, 'DD/MM/YYYY'),          -- mobile API sends day first
      TRY_TO_DATE(order_date, 'MON DD, YYYY')
    )                                                 AS order_date,
    TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
    UPPER(TRIM(currency))                             AS currency,
    TRIM(pincode)                                     AS pincode,
    TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
  FROM unified
)
SELECT *,
  ARRAY_CONSTRUCT_COMPACT(
    IFF(order_id IS NULL, 'missing_order_id', NULL),
    IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\s]+@[^@\\s]+\\.[a-z]{2,}$'), 'bad_email', NULL),
    IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
    IFF(order_date IS NULL, 'unparseable_date', NULL),
    IFF(updated_at IS NULL, 'unparseable_updated_at', NULL),
    IFF(amount <= 0, 'non_positive_amount', NULL)          -- NULL amount is allowed (flagged below)
  ) AS reasons
FROM typed;

-- Good rows
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT * EXCLUDE reasons, (amount IS NULL) AS amount_missing
FROM silver.orders_checked
WHERE ARRAY_SIZE(reasons) = 0;

-- Bad rows: kept with their reasons. The NOT EXISTS stops the same bad record being
-- quarantined again when a later run re-extracts it (same source, order, version).
INSERT INTO quarantine.orders (run_id, ingest_seq, source, order_id, reasons, row_data)
SELECT $run_id, c.ingest_seq, c.source, c.order_id, c.reasons, OBJECT_CONSTRUCT(*)
FROM silver.orders_checked c
WHERE ARRAY_SIZE(c.reasons) > 0
  AND NOT EXISTS (
        SELECT 1 FROM quarantine.orders q
        WHERE q.source = c.source
          AND EQUAL_NULL(q.order_id, c.order_id)
          AND EQUAL_NULL(q.row_data:UPDATED_AT::TIMESTAMP_NTZ, c.updated_at))   -- key is UPPERCASE: OBJECT_CONSTRUCT(*) names keys after the columns
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.source, c.order_id, c.updated_at
                           ORDER BY c.ingest_seq DESC) = 1;
"""

### SQL 20 ORDERS DEDUP

In [8]:
SQL_20_ORDERS_DEDUP = r"""
-- =====================================================================
-- SILVER 2: one row per order_id, newest version wins.
-- Ties on updated_at are broken by arrival order, so reruns give the same answer.
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_dedup AS
SELECT *
FROM silver.orders_clean
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY order_id
          ORDER BY updated_at DESC, ingest_seq DESC
        ) = 1;
"""

### SQL 30 CUSTOMERS SCD2

In [9]:
SQL_30_CUSTOMERS_SCD2 = r"""
-- =====================================================================
-- DIM: build the SCD Type 2 customer table from the CRM's tier-change log.
-- Each change starts a period; the period ends the day before the next change.
-- =====================================================================
CREATE OR REPLACE TABLE dim.customers_scd2 AS
WITH events AS (
  SELECT raw:cust_id::STRING                 AS cust_id,
         LOWER(TRIM(raw:email::STRING))      AS email,
         INITCAP(TRIM(raw:tier::STRING))     AS tier,
         TRY_TO_DATE(raw:effective_from::STRING) AS valid_from,
         loaded_at, ingest_seq
  FROM bronze.customers_raw
  -- the same event may be extracted on many runs; keep the latest copy
  QUALIFY ROW_NUMBER() OVER (PARTITION BY cust_id, valid_from
                             ORDER BY loaded_at DESC, ingest_seq DESC) = 1
)
SELECT cust_id, email, tier, valid_from,
       COALESCE(DATEADD(day, -1, LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from)),
                '9999-12-31'::DATE)                                       AS valid_to,
       LEAD(valid_from) OVER (PARTITION BY cust_id ORDER BY valid_from) IS NULL AS is_current
FROM events
WHERE cust_id IS NOT NULL AND valid_from IS NOT NULL;
"""

### SQL 31 FX DAILY

In [10]:
SQL_31_FX_DAILY = r"""
-- =====================================================================
-- REF: daily FX rates to INR, with no gaps.
-- 1. Flatten the API's {"rates": {"2026-09-01": {"INR": 83.2}, ...}} into rows.
-- 2. Rates are not published on weekends/holidays: fill each missing day with the
--    previous published rate, so every order date finds a rate.
-- 3. Add INR = 1.0 for every day, so rupee orders join like any other.
-- =====================================================================
CREATE OR REPLACE TABLE ref.fx_daily AS
WITH published AS (
  SELECT r.raw:base::STRING          AS currency,
         f.key::DATE                 AS rate_date,
         f.value:INR::NUMBER(12,4)   AS rate_to_inr
  FROM bronze.fx_raw r, LATERAL FLATTEN(input => r.raw:rates) f
  QUALIFY ROW_NUMBER() OVER (PARTITION BY currency, rate_date
                             ORDER BY r.loaded_at DESC, r.ingest_seq DESC) = 1
),
bounds AS (
  SELECT MIN(rate_date) AS d0, GREATEST(MAX(rate_date), CURRENT_DATE(), COALESCE((SELECT MAX(order_date) FROM silver.orders_dedup), CURRENT_DATE())) AS d1 FROM published
),
spine AS (                                   -- one row per calendar day d0..d1
  SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, b.d0) AS d
  FROM bounds b, TABLE(GENERATOR(ROWCOUNT => 4000))
  QUALIFY d <= (SELECT d1 FROM bounds)
),
filled AS (
  SELECT c.currency, s.d AS rate_date,
         LAST_VALUE(p.rate_to_inr) IGNORE NULLS OVER (
           PARTITION BY c.currency ORDER BY s.d
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rate_to_inr,
         p.rate_to_inr IS NULL AS is_carried_forward
  FROM spine s
  CROSS JOIN (SELECT DISTINCT currency FROM published) c
  LEFT JOIN published p ON p.currency = c.currency AND p.rate_date = s.d
)
SELECT currency, rate_date, rate_to_inr, is_carried_forward FROM filled
UNION ALL
SELECT 'INR', d, 1.0, FALSE FROM spine;
"""

### SQL 32 PINCODE GEO

In [11]:
SQL_32_PINCODE_GEO = r"""
-- =====================================================================
-- REF: add newly looked-up pincodes to ref.pincode_geo.
-- India Post gives district and state; the company's sales region comes from
-- the ref.state_region seed table.
-- =====================================================================
MERGE INTO ref.pincode_geo t
USING (
  SELECT p.pincode, p.city, p.state, sr.region
  FROM (
    SELECT raw:pincode::STRING                                  AS pincode,
           raw:response[0]:PostOffice[0]:District::STRING       AS city,
           raw:response[0]:PostOffice[0]:State::STRING          AS state,
           loaded_at, ingest_seq
    FROM bronze.pincode_raw
    WHERE raw:response[0]:Status::STRING = 'Success'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY pincode ORDER BY loaded_at DESC, ingest_seq DESC) = 1
  ) p
  LEFT JOIN ref.state_region sr ON sr.state = p.state
) s
ON t.pincode = s.pincode
WHEN MATCHED AND (t.city <> s.city OR t.state <> s.state OR NVL(t.region,'') <> NVL(s.region,''))
  THEN UPDATE SET city = s.city, state = s.state, region = s.region, updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED
  THEN INSERT (pincode, city, state, region, updated_at)
       VALUES (s.pincode, s.city, s.state, s.region, CURRENT_TIMESTAMP());
"""

### SQL 40 GOLD

In [12]:
SQL_40_GOLD = r"""
-- =====================================================================
-- ENRICH + PUBLISH
-- 1. silver.orders_enriched: point-in-time joins to customer tier and FX rate,
--    computed columns, geography. LEFT JOINs so no order is silently lost.
-- 2. gold.fct_orders: idempotent MERGE. Re-running changes nothing unless a
--    value actually changed (row_hash).
-- =====================================================================
CREATE OR REPLACE TABLE silver.orders_enriched AS
SELECT o.*,
       c.cust_id, c.tier,
       m.multiplier,
       fx.rate_to_inr                              AS fx_rate,
       ROUND(o.amount * fx.rate_to_inr, 2)         AS amount_inr,
       FLOOR(amount_inr / 100) * m.multiplier      AS loyalty_points,
       amount_inr >= 2000                          AS is_high_value,
       g.city, g.state, g.region
FROM silver.orders_dedup o
LEFT JOIN dim.customers_scd2 c
       ON  c.email = o.email
       AND o.order_date BETWEEN c.valid_from AND c.valid_to     -- tier ON the order date
LEFT JOIN ref.tier_multiplier m ON m.tier = c.tier
LEFT JOIN ref.fx_daily fx
       ON  fx.currency = o.currency AND fx.rate_date = o.order_date  -- rate OF the order date
LEFT JOIN ref.pincode_geo g ON g.pincode = o.pincode;

MERGE INTO gold.fct_orders t
USING (
  SELECT order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
         amount_inr, loyalty_points, is_high_value, city, state, region, updated_at,
         HASH(order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate,
              amount_inr, loyalty_points, is_high_value, city, state, region, updated_at) AS row_hash
  FROM silver.orders_enriched
) s
ON t.order_id = s.order_id
WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
  order_date = s.order_date, cust_id = s.cust_id, email = s.email, tier = s.tier,
  multiplier = s.multiplier, amount = s.amount, currency = s.currency, fx_rate = s.fx_rate,
  amount_inr = s.amount_inr, loyalty_points = s.loyalty_points, is_high_value = s.is_high_value,
  city = s.city, state = s.state, region = s.region, updated_at = s.updated_at,
  row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT
  (order_id, order_date, cust_id, email, tier, multiplier, amount, currency, fx_rate, amount_inr,
   loyalty_points, is_high_value, city, state, region, updated_at, row_hash, published_at)
VALUES
  (s.order_id, s.order_date, s.cust_id, s.email, s.tier, s.multiplier, s.amount, s.currency,
   s.fx_rate, s.amount_inr, s.loyalty_points, s.is_high_value, s.city, s.state, s.region,
   s.updated_at, s.row_hash, CURRENT_TIMESTAMP());
"""

## Orchestration and data checks
The FX calendar now includes the maximum order date so the intentionally future-dated B5 INR order can still be enriched. The sixth check is enabled only for B5. The baseline cleaning SQL stays unchanged until B3.

In [13]:
BASE_CLEAN_SQL = SQL_10_ORDERS_CLEAN
# Keep the baseline unchanged until the B3 experiment.
needle = "IFF(amount <= 0, 'non_positive_amount', NULL)"
assert needle in BASE_CLEAN_SQL
STRICT_CLEAN_SQL = BASE_CLEAN_SQL.replace(needle, needle + ",\n    IFF(currency IS NULL OR currency NOT IN ('INR','USD'), 'unsupported_currency', NULL)", 1)

BASE_CHECKS = {
 'enrichment row delta': 'SELECT (SELECT COUNT(*) FROM silver.orders_enriched)-(SELECT COUNT(*) FROM silver.orders_dedup)',
 'duplicate gold IDs': 'SELECT COUNT(*)-COUNT(DISTINCT order_id) FROM gold.fct_orders',
 'missing customer': 'SELECT COUNT_IF(cust_id IS NULL) FROM silver.orders_enriched',
 'missing FX for non-null amount': 'SELECT COUNT_IF(amount IS NOT NULL AND fx_rate IS NULL) FROM silver.orders_enriched',
 'missing city': 'SELECT COUNT_IF(city IS NULL) FROM silver.orders_enriched',
}
FUTURE_CHECK = 'SELECT COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup'

def quality_checks(extra=False):
    checks = dict(BASE_CHECKS)
    if extra: checks['future order date'] = FUTURE_CHECK
    result = []
    for name, sql in checks.items():
        value = int(query(sql)[0][0] or 0)
        result.append((name, value, 'OK' if value == 0 else 'PROBLEM'))
    frame = pd.DataFrame(result, columns=['check','violations','status'])
    rejected = query('SELECT COUNT_IF(ARRAY_SIZE(reasons)>0)/NULLIF(COUNT(*),0) FROM silver.orders_checked')[0][0]
    print('Rejected raw rows:', f'{float(rejected or 0):.1%}')
    display(frame)
    return frame


def gold_totals():
    return query('SELECT COUNT(*),SUM(amount_inr),SUM(loyalty_points) FROM gold.fct_orders')[0]


def snapshot():
    return show('SELECT order_id,tier,loyalty_points,row_hash,published_at FROM gold.fct_orders ORDER BY order_id').set_index('ORDER_ID')


def transform_loaded(run_id, strict=False):
    assert re.fullmatch(r'[A-Za-z0-9_]+', run_id)
    query(f"SET run_id = '{run_id}'")
    run_sql(STRICT_CLEAN_SQL if strict else BASE_CLEAN_SQL)
    run_sql(SQL_20_ORDERS_DEDUP)
    pins = [r[0] for r in query('SELECT DISTINCT o.pincode FROM silver.orders_dedup o WHERE NOT EXISTS (SELECT 1 FROM ref.pincode_geo g WHERE g.pincode=o.pincode)')]
    if pins:
        rows = [{'pincode': p, 'response': mock_pincode_response(p)} for p in pins]
        load_file(write_landing('pincode',rows,run_id),'bronze.pincode_raw',f'pincode/{run_id}')
    for sql in (SQL_30_CUSTOMERS_SCD2, SQL_31_FX_DAILY, SQL_32_PINCODE_GEO, SQL_40_GOLD):
        run_sql(sql)


def ingest_sources(web=None, crm=None, strict=False, extra=False):
    run_id = uuid4().hex
    sources = [('web', MOCK_WEB_ORDERS if web is None else web,'bronze.orders_raw'),
               ('mobile',MOCK_MOBILE_ORDERS,'bronze.orders_raw'),
               ('crm',MOCK_CRM_TIER_EVENTS if crm is None else crm,'bronze.customers_raw'),
               ('fx',MOCK_FX_RESPONSES,'bronze.fx_raw')]
    for source, records, table in sources:
        load_file(write_landing(source,records,run_id),table,f'{source}/{run_id}')
    transform_loaded(run_id,strict)
    print('Run:',run_id,'Gold totals:',gold_totals())
    return quality_checks(extra)


def layer_counts():
    tables = ['bronze.orders_raw','silver.orders_clean','quarantine.orders','silver.orders_dedup','gold.fct_orders']
    return pd.DataFrame([(t,query(f'SELECT COUNT(*) FROM {t}')[0][0]) for t in tables],columns=['table','rows'])

## B1 — live setup and baseline
Each new kernel run uses a dedicated database suffix, preventing accidental deletion or double counting in an existing database. The shared warehouse auto-suspends. Review resource usage and credentials before enabling live execution.

In [14]:
if RUN_LIVE:
    import os
    from dotenv import load_dotenv
    import snowflake.connector
    load_dotenv()
    def setting(name, default=None):
        value = os.getenv(name)
        if value: return value
        try:
            from google.colab import userdata
            return userdata.get(name) or default
        except Exception:
            return default
    auth = setting('SNOWFLAKE_AUTHENTICATOR','externalbrowser')
    conf = dict(account=setting('SNOWFLAKE_ACCOUNT'),user=setting('SNOWFLAKE_USER'),authenticator=auth)
    assert conf['account'] and conf['user'], 'Configure your own Snowflake secrets first'
    for key, env in [('role','SNOWFLAKE_ROLE'),('password','SNOWFLAKE_PASSWORD'),('private_key_file','SNOWFLAKE_PRIVATE_KEY_FILE'),('private_key_file_pwd','SNOWFLAKE_PRIVATE_KEY_FILE_PWD'),('token','SNOWFLAKE_TOKEN')]:
        value=setting(env)
        if value: conf[key]=value
    if conn is not None: conn.close()
    conn = snowflake.connector.connect(**conf)
    warehouse = setting('SNOWFLAKE_WAREHOUSE','FRESHCART_WH').upper()
    db_prefix = setting('SNOWFLAKE_DATABASE','FRESHCART').upper()
    assert re.fullmatch(r'[A-Z_][A-Z0-9_]{0,100}',warehouse)
    assert re.fullmatch(r'[A-Z_][A-Z0-9_]{0,100}',db_prefix)
    lab_db = db_prefix + '_LAB_' + uuid4().hex[:10].upper()
    setup = SQL_00_SETUP.replace('FRESHCART_WH',warehouse)
    setup = re.sub(r'\bFRESHCART\b',lab_db,setup)
    run_sql(setup)
    print('Isolated database:',lab_db)
    assert query('SELECT COUNT(*) FROM bronze.orders_raw')[0][0] == 0
    ingest_sources()
    display(layer_counts())
    display(show('SELECT order_id,reasons FROM quarantine.orders ORDER BY order_id'))
    assert list(layer_counts()['rows'])==[8,6,2,4,4]
    assert gold_totals()==(4,Decimal('7785.00'),126)
else:
    print('LIVE B1 NOT RUN — set RUN_LIVE=True in your own environment.')

LIVE B1 NOT RUN — set RUN_LIVE=True in your own environment.


## B2 — trace actual records
Use these queries and the original SQL to write your own explanation of normalization, newest-copy selection and historical tier joins.

In [15]:
if RUN_LIVE:
    display(show("SELECT source,raw FROM bronze.orders_raw WHERE COALESCE(raw:order_id::STRING,raw:orderId::STRING)='ORD-1003' ORDER BY ingest_seq"))
    display(show("SELECT source,order_id,order_date,amount,updated_at FROM silver.orders_checked WHERE order_id='ORD-1003'"))
    display(show("SELECT order_id,source,amount,updated_at FROM silver.orders_dedup WHERE order_id='ORD-1003'"))
    display(show("SELECT order_id,order_date,tier,multiplier,amount,fx_rate,amount_inr,loyalty_points FROM gold.fct_orders WHERE order_id IN ('ORD-1001','ORD-1003','ORD-1005') ORDER BY order_id"))

## B3 — add currency validation
The current checked table is recomputed; the first historical quarantine entry remains unchanged, because the insert anti-join uses source, order ID and source timestamp.

In [16]:
if RUN_LIVE:
    reason_sql="SELECT DISTINCT order_id, reasons FROM silver.orders_checked WHERE order_id='ORD-1006'"
    print('Before'); display(show(reason_sql))
    totals_before = gold_totals()
    old_quarantine = query('SELECT COUNT(*) FROM quarantine.orders')[0][0]
    ingest_sources(strict=True)
    print('After'); display(show(reason_sql))
    print('Historical quarantine'); display(show("SELECT order_id,reasons FROM quarantine.orders WHERE order_id='ORD-1006'"))
    assert totals_before == gold_totals()
    assert old_quarantine == query('SELECT COUNT(*) FROM quarantine.orders')[0][0]

## B4 — SCD2 and re-run safety
Record your five predictions in your own report before changing `B4_PREDICTIONS_RECORDED` to True. Compare the actual result rather than copying another report.

In [17]:
if RUN_LIVE and B4_PREDICTIONS_RECORDED:
    before = snapshot()
    before_bronze = query('SELECT COUNT(*) FROM bronze.orders_raw')[0][0]
    changed_crm = MOCK_CRM_TIER_EVENTS + [dict(cust_id='CUST-02',email='arjun.k@yahoo.in',tier='Gold',effective_from='2026-08-15')]
    ingest_sources(crm=changed_crm, strict=True)
    after = snapshot()
    display(show("SELECT * FROM dim.customers_scd2 WHERE cust_id='CUST-02' ORDER BY valid_from"))
    display(before.join(after,lsuffix='_before',rsuffix='_after'))
    print('Published timestamps changed:',int((before.PUBLISHED_AT!=after.PUBLISHED_AT).sum()))
    print('Bronze rows appended:',query('SELECT COUNT(*) FROM bronze.orders_raw')[0][0]-before_bronze)
    totals=gold_totals()
    bronze_before_replay=query('SELECT COUNT(*) FROM bronze.orders_raw')[0][0]
    ingest_sources(crm=changed_crm,strict=True)
    assert snapshot().equals(after) and gold_totals()==totals
    assert query('SELECT COUNT(*) FROM bronze.orders_raw')[0][0]-bronze_before_replay==8
    print('Unchanged re-run: gold rows and timestamps unchanged; bronze +8.')
else:
    print('B4 gated until you record predictions and enable live execution.')

B4 gated until you record predictions and enable live execution.


## B5 — future-date check and recovery
A uniquely named INR fixture is inserted and later removed only from this isolated database. A future order within the calendar-spine range is used so the new check, not missing FX, is the intentional failure.

In [18]:
if RUN_LIVE and B4_PREDICTIONS_RECORDED:
    assert (quality_checks(extra=True).violations==0).all()
    test_id='DQ-' + uuid4().hex
    future_day=query('SELECT DATEADD(day,1,GREATEST(CURRENT_DATE(),(SELECT MAX(order_date) FROM silver.orders_dedup)))')[0][0].isoformat()
    fixture=dict(order_id=test_id,cust_email='priya.r@gmail.com',cust_name='Priya R',order_date=future_day,amount='640.00',currency='INR',pincode='600041',updated_at=future_day+'T10:00:00')
    run_id=uuid4().hex
    previous=gold_totals()
    try:
        load_file(write_landing('web',[fixture],run_id),'bronze.orders_raw',f'dq/{run_id}')
        transform_loaded(run_id,strict=True)
        failed=quality_checks(extra=True)
        assert failed.loc[failed['check']=='future order date','violations'].iloc[0]==1
    finally:
        query('DELETE FROM bronze.orders_raw WHERE raw:order_id::STRING = %s',(test_id,))
        query('DELETE FROM gold.fct_orders WHERE order_id = %s',(test_id,))
        query('DELETE FROM quarantine.orders WHERE order_id = %s',(test_id,))
        transform_loaded(uuid4().hex,strict=True)
    assert (quality_checks(extra=True).violations==0).all()
    assert gold_totals()==previous
    print('Recovery verified:',gold_totals())

## Bonus — RabbitMQ batch to Gold
Run `python -m stream.lakehouse_consumer --seconds 60 --output bonus_batch.jsonl`, then in another terminal `python -m stream.dispatcher --freshcart --count 15`. Use the generated file in this notebook (upload to Colab if needed). The consumer persists JSONL before ACK; this notebook calls the same `write_landing`, `load_file`, and transformation SQL. Two replays get separate ingestion run IDs but unchanged order identities.

In [19]:
if RUN_LIVE and RUN_BONUS and B4_PREDICTIONS_RECORDED:
    batch=[json.loads(line) for line in Path(BONUS_BATCH_FILE).read_text(encoding='utf8').splitlines() if line.strip()]
    assert batch and all(r['order_id'].startswith('LIVE-') for r in batch)
    def replay(records):
        run_id=uuid4().hex
        load_file(write_landing('web',records,run_id),'bronze.orders_raw',f'bonus/{run_id}')
        transform_loaded(run_id,strict=True)
        assert (quality_checks(extra=True).violations==0).all()
        return gold_totals(),snapshot()
    initial=gold_totals()
    first,first_snapshot=replay(batch)
    second,second_snapshot=replay(batch)
    display(pd.DataFrame([initial,first,second],index=['before','first load','same batch replay'],columns=['orders','revenue','points']))
    assert first[0]>initial[0], 'Use a new batch with unique LIVE order IDs'
    assert first==second and first_snapshot.equals(second_snapshot)
    print('Replay verified: no duplicate gold totals or timestamp changes.')

## Close connection
The isolated Snowflake database is intentionally retained for inspection. Delete only your own lab database after reviewing evidence and confirming you no longer need it. No automatic DROP is performed.

In [20]:
if RUN_LIVE and conn is not None:
    conn.close()
    conn=None